# 03 Demand patterns and data quality

Labels every Texas product-store series by demand pattern, so later stages can ask where better forecasting pays off per pattern. Also checks the two data quality items flagged in Lesson 0: the largest daily sales value and prices under $0.10.

Inputs: `tx_weekly.parquet` and `tx_daily.parquet` from `02_join.ipynb`. Output: `series_profile.parquet`, one row per series.

## Kernel check

Confirms the notebook runs on the project venv, not Anaconda.

In [1]:
import sys, pandas, duckdb, pyarrow
print(sys.executable)
print(pandas.__version__, duckdb.__version__, pyarrow.__version__)

/Users/rajkandula/projects/walmart-demand-forecasting/.venv/bin/python
3.0.6 1.5.5 25.0.1


## Setup

In-memory DuckDB connection with views over the weekly and daily tables. Paths come from `src/config.py`.

In [2]:
import sys; sys.path.append("..")
import duckdb
from src.config import PROCESSED_DIR

con = duckdb.connect()
con.sql(f"""
CREATE OR REPLACE VIEW weekly AS
SELECT * FROM read_parquet('{PROCESSED_DIR / "tx_weekly.parquet"}')
""")
con.sql(f"""
CREATE OR REPLACE VIEW daily AS
SELECT * FROM read_parquet('{PROCESSED_DIR / "tx_daily.parquet"}')
""")

## Active weeks only

The join notebook showed no sales ever happen without a price. Zeros before an item's first priced week mean "not stocked yet," not "no demand," so they're dropped. A window function (`PARTITION BY id`) finds each series' first priced week without collapsing rows.

In [3]:
con.sql("""
CREATE OR REPLACE VIEW weekly_active AS
SELECT *
FROM (
    SELECT
        *,
        MIN(wm_yr_wk) FILTER (WHERE days_priced > 0)
            OVER (PARTITION BY id) AS first_priced_week
    FROM weekly
)
WHERE wm_yr_wk >= first_priced_week
""")

n_active, n_series, active_units = con.sql("""
SELECT COUNT(*), COUNT(DISTINCT id), SUM(sales)
FROM weekly_active
""").fetchone()
print(n_active, n_series, active_units)

assert n_series == 9_147
assert active_units == 19_199_909

2046387 9147 19199909


## Classify each series

Syntetos and Boylan method on weekly sales:
- **ADI**: average weeks per sale (active weeks divided by weeks with sales). Cutoff 1.32.
- **CV²**: squared coefficient of variation of nonzero weekly sales. Cutoff 0.49.

Low ADI and low CV² is smooth, low ADI and high CV² is erratic, high ADI and low CV² is intermittent, both high is lumpy. A series with only one nonzero week has no standard deviation, so its CV² is set to 0.

In [4]:
con.sql("""
CREATE OR REPLACE VIEW series_profile AS
WITH stats AS (
    SELECT
        id, item_id, dept_id, cat_id, store_id,
        MIN(first_priced_week) AS first_priced_week,
        COUNT(*) AS n_weeks,
        COUNT(*) FILTER (WHERE sales > 0) AS n_nonzero,
        AVG(sales) AS mean_weekly_sales,
        AVG(sales) FILTER (WHERE sales > 0) AS mean_nonzero,
        STDDEV_SAMP(sales) FILTER (WHERE sales > 0) AS sd_nonzero
    FROM weekly_active
    GROUP BY ALL
)
SELECT
    *,
    n_weeks / NULLIF(n_nonzero, 0) AS adi,
    COALESCE(POWER(sd_nonzero / mean_nonzero, 2), 0) AS cv2,
    CASE
        WHEN n_nonzero = 0 THEN 'no_sales'
        WHEN n_weeks / n_nonzero < 1.32
         AND COALESCE(POWER(sd_nonzero / mean_nonzero, 2), 0) < 0.49 THEN 'smooth'
        WHEN n_weeks / n_nonzero < 1.32 THEN 'erratic'
        WHEN COALESCE(POWER(sd_nonzero / mean_nonzero, 2), 0) < 0.49 THEN 'intermittent'
        ELSE 'lumpy'
    END AS pattern
FROM stats
""")

con.sql("""
SELECT pattern, COUNT(*) AS n_series,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_series,
       SUM(mean_weekly_sales * n_weeks)::BIGINT AS total_units,
       ROUND(100.0 * SUM(mean_weekly_sales * n_weeks)
             / SUM(SUM(mean_weekly_sales * n_weeks)) OVER (), 1) AS pct_units
FROM series_profile
GROUP BY pattern
ORDER BY n_series DESC
""").show()

┌──────────────┬──────────┬────────────┬─────────────┬───────────┐
│   pattern    │ n_series │ pct_series │ total_units │ pct_units │
│   varchar    │  int64   │   double   │    int64    │  double   │
├──────────────┼──────────┼────────────┼─────────────┼───────────┤
│ smooth       │     4402 │       48.1 │    14047843 │      73.2 │
│ intermittent │     3104 │       33.9 │     2173783 │      11.3 │
│ lumpy        │      942 │       10.3 │      771865 │       4.0 │
│ erratic      │      699 │        7.6 │     2206418 │      11.5 │
└──────────────┴──────────┴────────────┴─────────────┴───────────┘



## Pattern by store and category

`PIVOT` turns each pattern into a column: one row per store and category.

In [5]:
con.sql("""
PIVOT series_profile
ON pattern
USING COUNT(*)
GROUP BY store_id, cat_id
ORDER BY store_id, cat_id
""").show()

┌──────────┬───────────┬─────────┬──────────────┬───────┬────────┐
│ store_id │  cat_id   │ erratic │ intermittent │ lumpy │ smooth │
│ varchar  │  varchar  │  int64  │    int64     │ int64 │ int64  │
├──────────┼───────────┼─────────┼──────────────┼───────┼────────┤
│ TX_1     │ FOODS     │     168 │          370 │   193 │    706 │
│ TX_1     │ HOBBIES   │      35 │          273 │    84 │    173 │
│ TX_1     │ HOUSEHOLD │      36 │          452 │    52 │    507 │
│ TX_2     │ FOODS     │     154 │          361 │   147 │    775 │
│ TX_2     │ HOBBIES   │      42 │          202 │    78 │    243 │
│ TX_2     │ HOUSEHOLD │      48 │          354 │    69 │    576 │
│ TX_3     │ FOODS     │     161 │          404 │   172 │    700 │
│ TX_3     │ HOBBIES   │      29 │          229 │    91 │    216 │
│ TX_3     │ HOUSEHOLD │      26 │          459 │    56 │    506 │
└──────────┴───────────┴─────────┴──────────────┴───────┴────────┘



## Data quality: largest daily values

Lesson 0 found a 634-unit day against an average near 1 unit per product-store-day. Checking whether it's a one-off or part of a pattern.

In [6]:
con.sql("""
SELECT id, date, weekday, sales, sell_price, event_name_1, snap_TX
FROM daily
ORDER BY sales DESC
LIMIT 5
""").show()

┌─────────────────────────────────┬────────────┬───────────┬───────┬────────────┬───────────────┬─────────┐
│               id                │    date    │  weekday  │ sales │ sell_price │ event_name_1  │ snap_TX │
│             varchar             │    date    │  varchar  │ int16 │   double   │    varchar    │  int64  │
├─────────────────────────────────┼────────────┼───────────┼───────┼────────────┼───────────────┼─────────┤
│ FOODS_2_285_TX_1_evaluation     │ 2011-05-06 │ Friday    │   634 │       2.36 │ NULL          │       1 │
│ HOUSEHOLD_1_474_TX_2_evaluation │ 2013-07-24 │ Wednesday │   626 │       0.97 │ NULL          │       0 │
│ HOUSEHOLD_1_474_TX_2_evaluation │ 2013-10-16 │ Wednesday │   607 │       0.97 │ NULL          │       0 │
│ HOUSEHOLD_1_474_TX_2_evaluation │ 2014-02-17 │ Monday    │   606 │       0.97 │ PresidentsDay │       0 │
│ HOUSEHOLD_2_062_TX_1_evaluation │ 2013-06-05 │ Wednesday │   601 │       2.94 │ NULL          │       1 │
└───────────────────────────

## Data quality: prices under $0.10

First contact found a minimum price of $0.01. Listing every item-store with prices under $0.10.

In [7]:
con.sql("""
SELECT item_id, store_id, MIN(sell_price) AS min_price, MAX(sell_price) AS max_price,
       COUNT(*) AS days_under_10c, SUM(sales)::INTEGER AS units_under_10c
FROM daily
WHERE sell_price < 0.10
GROUP BY ALL
ORDER BY days_under_10c DESC
""").show()

┌───────────────┬──────────┬───────────┬───────────┬────────────────┬─────────────────┐
│    item_id    │ store_id │ min_price │ max_price │ days_under_10c │ units_under_10c │
│    varchar    │ varchar  │  double   │  double   │     int64      │      int32      │
├───────────────┼──────────┼───────────┼───────────┼────────────────┼─────────────────┤
│ FOODS_3_711   │ TX_3     │      0.05 │      0.08 │            119 │             698 │
│ FOODS_3_580   │ TX_3     │      0.05 │      0.08 │             98 │             960 │
│ FOODS_3_007   │ TX_3     │      0.05 │      0.08 │             84 │            1071 │
│ FOODS_3_547   │ TX_3     │      0.05 │      0.08 │             84 │            1037 │
│ FOODS_3_070   │ TX_3     │      0.05 │      0.09 │             70 │             295 │
│ FOODS_3_412   │ TX_3     │      0.05 │      0.09 │             70 │             713 │
│ FOODS_3_454   │ TX_3     │      0.05 │      0.08 │             70 │             452 │
│ FOODS_3_752   │ TX_2     │    

## Save the series profile

Used by the forecasting notebook (pattern as a feature and a reporting slice) and by the Power BI dashboard.

In [8]:
profile_path = PROCESSED_DIR / "series_profile.parquet"
con.sql(f"COPY series_profile TO '{profile_path}' (FORMAT parquet)")
n_profile = con.sql(f"SELECT COUNT(*) FROM read_parquet('{profile_path}')").fetchone()[0]
print(n_profile)
assert n_profile == 9_147

9147


## Results

**Active weeks.** 2,046,387 of 2,533,719 weekly rows are on or after each series' first priced week. The 487,332 dropped rows (19.2%) held zero units, so all 19,199,909 units are kept.

**Demand patterns.**

| Pattern | Series | % of series | Units | % of units |
|---|---|---|---|---|
| Smooth | 4,402 | 48.1 | 14,047,843 | 73.2 |
| Intermittent | 3,104 | 33.9 | 2,173,783 | 11.3 |
| Erratic | 699 | 7.6 | 2,206,418 | 11.5 |
| Lumpy | 942 | 10.3 | 771,865 | 4.0 |

No series had zero sales. Smooth series carry almost three quarters of volume. Intermittent and lumpy series are 44.2% of series but 15.3% of units, so forecast improvements there touch little volume. Stage 3 tests whether a more complex model still earns its keep on them.

**By store and category.** All three stores show the same shape. HOUSEHOLD has the most intermittent series in every store (354 to 459). FOODS has the most smooth, erratic, and lumpy series.

**Largest daily values.** The 634-unit day is FOODS_2_285 at TX_1 on Friday 2011-05-06, a SNAP day with no event, at $2.36. HOUSEHOLD_1_474 at TX_2 holds three of the top five days (606 to 626 units at $0.97). These look like bulk purchases rather than recording errors, so they stay in. They will show up as large single-week errors for any method.

**Prices under $0.10.** 35 item-store combinations, almost all FOODS_3, priced $0.02 to $0.09, some selling over 1,000 units at those prices. Likely a pricing data quirk rather than real shelf prices. Kept, but flagged: in the cost analysis they carry near-zero dollar value and can't distort totals.